In [30]:
# Download data dan src dari GitHub (uncomment jika di Colab)
!rm -rf IR-BERT-Transformer data src
!git clone https://github.com/teranixbq/IR-BERT-Transformer.git
!mv IR-BERT-Transformer/data ./data
!mv IR-BERT-Transformer/src ./src
!rm -rf IR-BERT-Transformer

Cloning into 'IR-BERT-Transformer'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 56 (delta 24), reused 45 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 28.34 KiB | 14.17 MiB/s, done.
Resolving deltas: 100% (24/24), done.


# Neural IR Exercise dengan BERT Cross-Encoder
Dosen: Zico Pratama Putra
Kelompok: [Nama 1] - [Nama 2] - [Nama 3]
Tanggal: ---

In [31]:
from src.judgement_aggregation import load_raw_judgements, aggregate_judgements, save_qrels, compute_annotator_reliability
from src.bert_cross_encoder import BERTCrossEncoder
from transformers import pipeline
import pandas as pd

DEST = "/content/data/"
RAW_JUDGEMENTS = DEST + "fira_raw_judgements.tsv"

## Load Raw

In [34]:
df = load_raw_judgements(RAW_JUDGEMENTS)

Loaded 237 raw judgements
Columns: ['query_id', 'doc_id', 'judgement', 'confidence', 'annotator_id', 'duration_ms']
   query_id doc_id  judgement  confidence annotator_id  duration_ms
0         1   d1_1          3        0.85       User_5        30289
1         1   d1_1          3        0.94       User_0        83482
2         1   d1_1          3        0.92       User_4        16165
3         1   d1_2          2        0.81       User_0        38062
4         1   d1_2          1        0.92       User_5        12851


# PART 1 : FiRA Judgement Aggregation

## 1.1 Baseline - Simple Majority Vote

In [41]:
agg_maj = aggregate_judgements(df, method='majority')
agg_maj.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,std_score
0,1,d1_1,3,3,0.903
1,1,d1_2,2,3,0.867
2,1,d1_3,1,3,0.700
3,1,d1_4,0,3,0.953
4,1,d1_5,0,3,0.907
5,1,d1_6,0,3,0.967
6,1,d1_7,1,3,0.780
7,1,d1_8,1,3,0.840
8,2,d2_1,3,3,0.937
9,2,d2_2,2,3,0.887


## 1.2 Confidence-Based Weighting
Bobot berdasarkan seberapa yakin annotator

In [36]:
agg_weighted = aggregate_judgements(df, method='advanced', strategy='confidence_weighted')
agg_weighted.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,std_score
0,1,d1_1,3,3,0.903
1,1,d1_2,2,3,0.867
2,1,d1_3,1,3,0.700
3,1,d1_4,0,3,0.953
4,1,d1_5,0,3,0.907
5,1,d1_6,0,3,0.967
6,1,d1_7,1,3,0.780
7,1,d1_8,1,3,0.840
8,2,d2_1,3,3,0.937
9,2,d2_2,2,3,0.887


## 1.3 Reliability-Weighted Voting
Bobot dari track record annotator — seberapa sering ia setuju dengan suara terbanyak (majority vote). Annotator yang sering benar dapat bobot lebih besar.

In [38]:
reliability = compute_annotator_reliability(df)
agg_reliability = aggregate_judgements(df, method='advanced', strategy='annotator_reliability', annotator_reliability=reliability)
agg_reliability.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,std_score
0,1,d1_1,3,3,0.903
1,1,d1_2,2,3,0.867
2,1,d1_3,1,3,0.700
3,1,d1_4,0,3,0.953
4,1,d1_5,0,3,0.907
5,1,d1_6,0,3,0.967
6,1,d1_7,1,3,0.780
7,1,d1_8,1,3,0.840
8,2,d2_1,3,3,0.937
9,2,d2_2,2,3,0.887


## 1.4 Simpan Semua

In [ ]:
save_qrels(agg_maj, DEST + "fira_aggregated.qrels")
save_qrels(agg_weighted, DEST + "fira_aggregated_confidence_weighted.qrels")
save_qrels(agg_reliability, DEST + "fira_aggregated_annotator_reliability.qrels")

## Perbandingan BASELINE VS ADVANCE

In [39]:
print("=== PERBANDINGAN BASELINE (MAJORITY) VS ADVANCED ===")
print(f"{'Method':<30} {'Score Dist':<30} {'Mean Std Score':<15}")
print("-" * 75)
for name, result in [('majority (baseline)', agg_maj),
                       ('confidence_weighted', agg_weighted),
                       ('annotator_reliability', agg_reliability)]:
    dist = result['score'].value_counts().sort_index().to_dict()
    mean_std = result['std_score'].mean()
    print(f"{name:<30} {str(dist):<30} {mean_std:<15.3f}")

=== PERBANDINGAN BASELINE (MAJORITY) VS ADVANCED ===
Method                         Score Dist                     Mean Std Score 
---------------------------------------------------------------------------
majority (baseline)            {0: 34, 1: 24, 2: 10, 3: 11}   0.861          
confidence_weighted            {0: 34, 1: 24, 2: 10, 3: 11}   0.861          
annotator_reliability          {0: 34, 1: 24, 2: 10, 3: 11}   0.861          


## Analisis Manual

In [40]:
print("\n=== ANALISIS MANUAL: Contoh Query-Doc Pair ===")
sample_pairs = [(1, 'd1_3'), (1, 'd1_8'), (2, 'd2_1')]
for qid, did in sample_pairs:
    raw = df[(df['query_id'] == qid) & (df['doc_id'] == did)]
    print(f"\nQuery {qid}, Doc {did}:")
    for _, row in raw.iterrows():
        rel = reliability.get(row['annotator_id'], 0)
        print(f"  {row['annotator_id']}: judgement={row['judgement']}, confidence={row['confidence']:.3f}, reliability={rel:.3f}")
    maj_score = agg_maj[(agg_maj['query_id'] == qid) & (agg_maj['doc_id'] == did)]['score'].values[0]
    w_score = agg_weighted[(agg_weighted['query_id'] == qid) & (agg_weighted['doc_id'] == did)]['score'].values[0]
    r_score = agg_reliability[(agg_reliability['query_id'] == qid) & (agg_reliability['doc_id'] == did)]['score'].values[0]
    print(f"  -> majority={maj_score}, confidence_weighted={w_score}, annotator_reliability={r_score}")


=== ANALISIS MANUAL: Contoh Query-Doc Pair ===

Query 1, Doc d1_3:
  User_1: judgement=1, confidence=0.870, reliability=0.837
  User_2: judgement=2, confidence=0.650, reliability=0.780
  User_0: judgement=1, confidence=0.580, reliability=0.769
  -> majority=1, confidence_weighted=1, annotator_reliability=1

Query 1, Doc d1_8:
  User_4: judgement=1, confidence=0.870, reliability=0.828
  User_2: judgement=1, confidence=0.820, reliability=0.780
  User_3: judgement=0, confidence=0.830, reliability=0.766
  -> majority=1, confidence_weighted=1, annotator_reliability=1

Query 2, Doc d2_1:
  User_5: judgement=2, confidence=0.880, reliability=0.763
  User_3: judgement=3, confidence=0.990, reliability=0.766
  User_0: judgement=3, confidence=0.940, reliability=0.769
  -> majority=3, confidence_weighted=3, annotator_reliability=3


## Part 2: BERT Cross-Encoder Re-Ranking

In [ ]:
reranker = BERTCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

query = "How to make a good cappuccino?"
passages = ["Passage 1 ...", "Passage 2 ...", "Passage 3 ..."]
ranked_idx, scores = reranker.re_rank(query, passages)
print("Ranked order:", ranked_idx)

## Part 3: Extractive QA

In [ ]:
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

result = qa_pipeline(question=query, context=passages[ranked_idx[0]])
print(result)

## Evaluasi
# TODO: Implement evaluasi MRR@10, NDCG@10, dll.